# Milk Delivery for Hawker Centers in Singapore

Jack is a driver of ABC Milk Company. He is assigned to deliver fresh milk for 20 hawker centers each morning.  Suppose he can visit each center only once, could you help Jack to design a rounte with least amount of time (distance)?

The geolocation data of the hawker centers are imported as follows:

## Improt packages

**You may need to install the following packages: folium, haversine, openrouteservice.**

You can do so by command "pip install xxxxxx" in your terminal, check the installation guide I uploaded before the first lecture of this course.

In [1]:
import pandas as pd
import numpy as np

import folium                             ## Package for Plotting on Map
from folium.plugins import BeautifyIcon
#import mpu                                
from haversine import haversine as haversine_distance   ## Package for checking haversine distances
import openrouteservice as ors            ## Package for checking driving distances

from rsome import ro
#from rsome import ort_solver as ort
from rsome import lpg_solver as lpg

C:\Users\Ozean\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


## Read data and visualize locations

In [2]:
## Import the geolocation data of all the hawker centres
data = pd.read_csv('hawker_centres_kml.csv')
# print(data.head(5))

In [3]:
lon = data['X']
lat = data['Y']

N = 10
lon = [lon[i] for i in range(N)]
lat = [lat[i] for i in range(N)]


## plot the locations in singapore map
# https://livecodestream.dev/post/how-to-plot-your-data-on-maps-using-python-and-folium/
SGmap = folium.Map(location=[lat[2], lon[2]], zoom_start=11) #Stamen Terrain, OpenStreetMap
for i in range(N):
    folium.Marker(
        location=[lat[i], lon[i]],
        icon=BeautifyIcon(
            icon_shape='marker',
            number=int(i + 1),
            spin=True,
            text_color='red',
            background_color="#FFF",
            inner_icon_style="font-size:12px;padding-top:-5px;"
        )
    ).add_to(SGmap)
SGmap

### Prepare the distance matrix

In [4]:
haversine_distance((lat[0], lon[0]),(lat[1], lon[1]))

14.275151342476821

In [5]:
distance_mat = np.array([[haversine_distance((lat[i], lon[i]), (lat[j], lon[j]))
                          for j in range(N)] 
                          for i in range(N)])
print(distance_mat)

[[ 0.         14.27515134 12.9997454   8.72155098 11.52261923  6.79890519
  11.97672696 24.14817814  1.65335149 16.06525332]
 [14.27515134  0.          9.53149192 10.00187663  7.48592453  7.55294069
   2.81878142 11.9773084  15.46353165  2.7461948 ]
 [12.9997454   9.53149192  0.          4.31932088  2.43567722  8.32057567
  10.14745482 12.66491271 13.19345567  8.69128413]
 [ 8.72155098 10.00187663  4.31932088  0.          3.52726606  5.31353572
   9.37153468 16.45258208  8.87463658 10.34986031]
 [11.52261923  7.48592453  2.43567722  3.52726606  0.          6.10583797
   7.78835926 12.98606951 11.98140543  7.18504133]
 [ 6.79890519  7.55294069  8.32057567  5.31353572  6.10583797  0.
   5.58971343 17.66028769  7.91376049  9.27116621]
 [11.97672696  2.81878142 10.14745482  9.37153468  7.78835926  5.58971343
   0.         14.72394242 13.29986856  5.46800545]
 [24.14817814 11.9773084  12.66491271 16.45258208 12.98606951 17.66028769
  14.72394242  0.         24.83176677  9.25760845]
 [ 1.653

## Model formulation: the first try

**(Baseline formulation)** Label the centres with the numbers $1,\ldots,N$ and define:
$$
x_{ij} = \begin{cases} 
1 & \text{the path goes from centre } i \text{ to centre } j \\ 
0 & \text{otherwise} 
\end{cases}
$$

For $i=1,\ldots,N$, let $u_i$ be a dummy variable, and finally take $d_{ij} > 0$ to be the distance from centre $i$ to centre $j$. Then TSP can be written as the following integer linear programming problem:
$$
\begin{align}
\min &\sum_{i=1}^n \sum_{j=1}^n d_{ij}x_{ij}\colon &&  \\
     & \sum_{i=1}^N x_{ij} = 1 && j=1, \ldots, N; \\
     & \sum_{j=1}^N x_{ij} = 1 && i=1, \ldots, N; \\
     & x_{ij} \in \{0,1\}  && i,j=1, \ldots, N; \\
     & x_{ii} = 0  && i= 1, \ldots, N; \\
\end{align}
$$

In [6]:
def Hawker_Centre_TSP_baseline(N, distance):
    ## Miller–Tucker–Zemlin formulation
    m = ro.Model('Hawker Centre TSP-MTZ')
    x = m.dvar((N,N),'B')
    u = m.dvar(N)
    
    ########## Fill up your code here to complete the optimization model  ##########
    m.min(sum(distance[i, j] * x[i, j] for i in range(N) for j in range(N)))
    for j in range(N):
        m.st(sum(x[i, j] for i in range(N)) == 1)
    for i in range(N):
        m.st(sum(x[i, j] for j in range(N)) == 1)
    for _ in range(N):
        m.st(x[_, _] == 0)
    ################################################################################

    m.solve(lpg)
    x_star = x.get()
    obj = m.get()
    return x_star, obj

There are two parameters for this model: number of centers $N$ and distrance matrix $\boldsymbol{d}$.

In [7]:
x_baseline, obj_baseline = Hawker_Centre_TSP_baseline(N, distance_mat[0:N,0:N])

Being solved by the default MILP solver...
Solution status: 0
Running time: 0.0186s


In [8]:
print(x_baseline)

[[ 0.  0.  0.  0.  0.  0.  0.  0.  1.  0.]
 [ 0.  0.  0.  0.  0.  0.  1. -0.  0.  0.]
 [ 0.  0.  0. -0.  1.  0.  0.  0.  0.  0.]
 [ 0.  0. -0.  0. -0.  1.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0. -0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  1.]
 [ 1.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  1.  0.  0.]]


In [9]:
ind = np.argwhere( x_baseline )
print(ind)

[[0 8]
 [1 6]
 [2 4]
 [3 5]
 [4 2]
 [5 3]
 [6 1]
 [7 9]
 [8 0]
 [9 7]]


### Obtain the routes given by the optimal solution

In [10]:
## obtain [lon, lat] pairs for the route
def obtain_route(x_star, lon, lat):
    N = len(lon)
    seq_route = -np.ones(N + 1) # initialize some numbers for the route sequence
    lon_route = -np.ones(N + 1)
    lat_route = -np.ones(N + 1)

    # starting point is food center 1
    lon_route[0] = lon[0]
    lat_route[0] = lat[0]
    seq_route[0]= 1

    ind = np.argwhere( x_star>0.1 )   # find where x=1
    
    ind_j = 0                     # The first stop
    for i in range(N):
        ind_j = ind[ind_j,1]      # Check the next stop
        seq_route[i+1] = ind_j + 1
        lon_route[i+1] = lon[ind_j]
        lat_route[i+1] = lat[ind_j]
        
    return seq_route, lon_route, lat_route

In [11]:
seq_route_tsp, lon_route_tsp, lat_route_tsp = obtain_route(x_baseline, lon[0:N], lat[0:N])
seq_route_tsp

array([1., 9., 1., 9., 1., 9., 1., 9., 1., 9., 1.])

 **This does not seem to be correct... Is our code wrong? Is our model wrong?**

## Model formulation: Miller–Tucker–Zemlin formulation


**(Miller–Tucker–Zemlin formulation)** Label the centres with the numbers $1,\ldots,N$ and define:
$$
x_{ij} = \begin{cases} 
1 & \text{the path goes from centre } i \text{ to centre } j \\ 
0 & \text{otherwise} 
\end{cases}
$$


In [15]:
def Hawker_Centre_TSP_MTZ(N, distance):
    ## Miller–Tucker–Zemlin formulation
    m = ro.Model('Hawker Centre TSP-MTZ')
    x = m.dvar((N,N),'B')
    u = m.dvar(N)

    ########## Fill up your code here to complete the optimization model  ##########
    m.min(sum(distance[i, j] * x[i, j] for i in range(N) for j in range(N)))
    for j in range(N):
        m.st(sum(x[i, j] for i in range(N)) == 1)
    for i in range(N):
        m.st(sum(x[i, j] for j in range(N)) == 1)
    for _ in range(1, N):
        m.st(x[_, _] == 0)
    for _ in range(1, N):
        m.st(u[_] >= 0)
        m.st(u[_] <= N - 1)
    for i in range(1, N):
        for j in range(1, N):
            if i != j:
                m.st(u[i] - u[j] + N * x[i, j] <= N -1)
    m.st(u[0] == 0)

    ################################################################################

    m.solve(lpg)
    x_star = x.get()
    obj = m.get()
    return x_star, obj

Now you can solve the model to obtain the solution:

In [16]:
########## Fill up your code here to obtain the optimal solution  ##########
x_tsp, obj_tsp = Hawker_Centre_TSP_MTZ(N, distance_mat[0:N,0:N])
############################################################################

Being solved by the default MILP solver...
Solution status: 0
Running time: 0.6414s


### Obtain and visualize the routes given by the optimal solution

You can obtain the route via the following function:

In [18]:
########## Fill up your code here to complete this line  ##########
seq_route_tsp, lon_route_tsp, lat_route_tsp =  obtain_route(x_tsp, lon[0:N], lat[0:N])
seq_route_tsp
###################################################################

array([ 1.,  9.,  4.,  5.,  3.,  8., 10.,  2.,  7.,  6.,  1.])

Now you can plot the (approximate) routes on the map:

In [19]:
for i in range(N):
    folium.PolyLine([[lat_route_tsp[i], lon_route_tsp[i]], [lat_route_tsp[i + 1], lon_route_tsp[i + 1]]]).add_to(SGmap)
SGmap

### Some issues with the solution:

1. Distance is an approximation;

2. Routes are not on the real roads.

You can use `openrouteservice` to estimate the distance matrix and also use this package to draw the real routes for the example.

To use `openrouteservice`, you need to sign up in https://openrouteservice.org/dev/#/signup and request an API Key.

In [32]:
## Import the geolocation data of all the hawker centres
data = pd.read_csv('hawker_centres_kml.csv')
# print(data.head(5))

## obtain the distance matrix
lon = data['X']
lat = data['Y']
N = 10 ## maximum locations = 50
lon = [lon[i] for i in range(N)]
lat = [lat[i] for i in range(N)]
latlon = [[lat[i], lon[i]] for i in range(N)]
lonlat = [[lon[i], lat[i]] for i in range(N)]

import requests
body = {"locations":lonlat[0:50],"metrics":["distance"]} ## maximum 50 locations

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': '5b3ce3597851110001cf62481d413a72b15a48288beb35d5241767c6',
    'Content-Type': 'application/json; charset=utf-8'
}
call = requests.post('https://api.openrouteservice.org/v2/matrix/driving-car', json=body, headers=headers)
print(call.status_code, call.reason)
data_req = call.json()


distance_mat = np.array(data_req['distances'])/1000  # Obtain an updated distance matrix using actual traffic distance

ConnectTimeout: HTTPSConnectionPool(host='api.openrouteservice.org', port=443): Max retries exceeded with url: /v2/matrix/driving-car (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.openrouteservice.org', port=443) at 0x1dec395efd0>, 'Connection to api.openrouteservice.org timed out. (connect timeout=None)'))

In [25]:
## plot the selected locations in a new map
TSPmap1 = folium.Map(location=[lat[2], lon[2]], tiles="OpenStreetMap", zoom_start=11.5) #Stamen Terrain, OpenStreetMap
for i in range(N):
    folium.Marker(
        location=[lat[i], lon[i]],
        icon=BeautifyIcon(
            icon_shape='marker',
            number=int(i + 1),
            spin=True,
            text_color='red',
            background_color="#FFF",
            inner_icon_style="font-size:12px;padding-top:-5px;"
        )
    ).add_to(TSPmap1)
TSPmap1

In [26]:
########## Fill up your code here to complete this line  ##########
# obtain the best routes using the updated distance matrix
x_tsp, obj_tsp =  Hawker_Centre_TSP_MTZ(N, distance_mat[0:N,0:N])
###################################################################


Being solved by the default MILP solver...
Solution status: 0
Running time: 0.6106s


In [27]:
seq_route_tsp, lon_route_tsp, lat_route_tsp = obtain_route(x_tsp, lon[0:N], lat[0:N])

In [31]:
lon_route = lon_route_tsp
lat_route = lat_route_tsp
for i in range(N):
    # Specify your personal API key, registered in openrouteservice.com
    client = ors.Client(key = '5b3ce3597851110001cf62488317235a3dc24e77b257d920e034b0fc') 
    coordinates = [[lon_route[i], lat_route[i]], [lon_route[i + 1], lat_route[i + 1]]]

    route = client.directions(coordinates = coordinates, profile = 'foot-walking', format = 'geojson') #driving-car, cycling-road, cycling-regular, foot-walking
    
    gj = folium.GeoJson(
        name='route',
        data={"type": "FeatureCollection", "features": [{"type": "Feature",
                                                         "geometry": route['features'][0]['geometry']}]},
    )
    
    gj.add_to(TSPmap1)

TSPmap1

Timeout: 